# Week 2 Automated Preprocessing Pipeline

This notebook:

1. Reads `week1_initial_raw_dataset.csv`
2. Cleans and validates the data
3. Creates Week 2 financial features
4. Saves `week2_feature_dataset_pipeline.csv`
5. Can be executed automatically by GitHub Actions

**GitHub repository layout**

```text
Chooser-Option-Pricing/
├── preprocessing.ipynb
├── week1_initial_raw_dataset.csv
├── requirements.txt
└── .github/
    └── workflows/
        └── pipeline.yml
```


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)


In [ ]:
def build_week2_pipeline(
    input_file: str,
    output_file: str,
    rolling_window: int = 20,
) -> pd.DataFrame:
    """
    Read the Week 1 raw dataset, clean it, create Week 2 features,
    and save the processed dataset.

    Parameters
    ----------
    input_file:
        Path to the Week 1 CSV file.
    output_file:
        Path where the processed Week 2 CSV will be saved.
    rolling_window:
        Window used for rolling volatility and correlation.

    Returns
    -------
    pd.DataFrame
        The processed Week 2 feature dataset.
    """

    input_path = Path(input_file)
    output_path = Path(output_file)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Input file was not found: {input_path.resolve()}\n"
            "Make sure week1_initial_raw_dataset.csv is in the same "
            "GitHub repository folder as this notebook."
        )

    df = pd.read_csv(input_path)
    input_rows = len(df)

    # Standardize column names.
    df.columns = [
        str(column).strip().replace(" ", "_")
        for column in df.columns
    ]

    # Support either Adj Close or Adj_Close.
    if "Adj_Close" not in df.columns and "Adj Close" in df.columns:
        df = df.rename(columns={"Adj Close": "Adj_Close"})

    required_columns = {"Close"}
    missing_required = required_columns.difference(df.columns)

    if missing_required:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(sorted(missing_required))
        )

    # Parse and sort dates.
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"])
        df = df.sort_values("Date")
        df = df.drop_duplicates(subset=["Date"], keep="last")

    # Convert market columns to numeric.
    numeric_columns = [
        "Open",
        "High",
        "Low",
        "Close",
        "Adj_Close",
        "Volume",
        "VIX",
        "Treasury_10Y",
    ]

    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    # Forward-fill market observations.
    available_numeric_columns = [
        column for column in numeric_columns if column in df.columns
    ]
    df[available_numeric_columns] = (
        df[available_numeric_columns].ffill()
    )

    # Core return features.
    df["Daily_Return"] = df["Close"].pct_change()
    df["Log_Return"] = np.log(
        df["Close"] / df["Close"].shift(1)
    )

    # Annualized historical volatility.
    df["Rolling_Volatility_20D"] = (
        df["Log_Return"]
        .rolling(window=rolling_window)
        .std()
        * np.sqrt(252)
    )

    # VIX features.
    if "VIX" in df.columns:
        df["VIX_Change"] = df["VIX"].pct_change()

        df["VIX_JPM_Correlation_20D"] = (
            df["Daily_Return"]
            .rolling(window=rolling_window)
            .corr(df["VIX_Change"])
        )

    # Interest-rate feature.
    if "Treasury_10Y" in df.columns:
        df["Rate_Change"] = df["Treasury_10Y"].diff()

    # Volume feature.
    if "Volume" in df.columns:
        df["Volume_Change"] = df["Volume"].pct_change()

    # Replace infinite values caused by percentage changes.
    df = df.replace([np.inf, -np.inf], np.nan)

    # Remove the initial rows that cannot have rolling features.
    required_features = [
        "Daily_Return",
        "Log_Return",
        "Rolling_Volatility_20D",
    ]
    df = df.dropna(subset=required_features).reset_index(drop=True)

    # Create the output directory and save the CSV.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)

    print("Pipeline completed successfully.")
    print(f"Input rows:  {input_rows}")
    print(f"Output rows: {len(df)}")
    print(f"Output file: {output_path.resolve()}")

    return df


## Run the pipeline

These are **relative GitHub paths**, not Google Drive or `/content/drive` paths.
The generated CSV is saved in the GitHub Actions working directory.
The workflow file then commits it back to the repository.


In [ ]:
INPUT_FILE = "data/week1_initial_raw_dataset.csv"
OUTPUT_FILE = "output/week2_feature_dataset_pipeline.csv"

week2_data_pipeline = build_week2_pipeline(
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
)


In [ ]:
display(week2_data_pipeline.head())
display(week2_data_pipeline.tail())


In [ ]:
quality_report = pd.DataFrame({
    "Column": week2_data_pipeline.columns,
    "Data_Type": week2_data_pipeline.dtypes.astype(str).values,
    "Missing_Values": week2_data_pipeline.isna().sum().values,
    "Unique_Values": week2_data_pipeline.nunique().values,
})

display(quality_report)


In [ ]:
output_path = Path(OUTPUT_FILE)

assert output_path.exists(), (
    f"The expected output file was not generated: "
    f"{output_path.resolve()}"
)
assert not week2_data_pipeline.empty, (
    "The processed dataset is empty."
)

print(
    f"Verified: {OUTPUT_FILE} exists and contains "
    f"{len(week2_data_pipeline)} rows."
)


## How the file is saved to GitHub

This notebook creates `week2_feature_dataset_pipeline.csv` while the
GitHub Action is running. The accompanying `pipeline.yml` performs a
Git commit and push after the notebook finishes.

After a successful run, return to the repository's **Code** page and
you will see:

```text
week2_feature_dataset_pipeline.csv
```
